In [2]:
import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 6 전처리 최종 검증")
print("=" * 70)

# 전처리 완료본
df = pd.read_csv("../../data/processed/2_preprocess/step5_weather.csv", low_memory=False)
df["BIRTH_YMD"]  = pd.to_datetime(df["BIRTH_YMD"], errors="coerce")
df["ABATT_DATE"] = pd.to_datetime(df["ABATT_DATE"], errors="coerce")
df["JUDGE_DATE"] = pd.to_datetime(df["JUDGE_DATE"], errors="coerce")

# 원본 (무손상 대조용)
raw_train   = pd.read_csv("../../data/raw/hanwoo_train.csv")
raw_lineage = pd.read_csv("../../data/raw/hanwoo_lineage.csv", low_memory=False)
raw_death   = pd.read_csv("../../data/raw/hanwoo_death.csv")
raw_area    = pd.read_csv("../../data/raw/hanwoo_area.csv")

results = []
def check(name, condition):
    results.append(bool(condition))
    print(f"  [{'PASS' if condition else 'FAIL'}] {name}")
    return condition

# =====================================================
# 섹션 0. 전처리 완료 검증 (-99 / 0 / 더미 제거)
# =====================================================
print("\n[섹션 0] 전처리 완료 검증")

# 수치형 -99 완전 제거
num_check_cols = ["BACKFAT","REA","WINDEX","INSFAT","YUKSAK","FATSAK","TISSUE",
                  "GROWTH","COST_AMT","WEIGHT","AGE","death_count",
                  "C2023","C2024","C2025","AREA"]
no_minus99 = True
for col in num_check_cols:
    if (df[col] == -99).sum() > 0:
        no_minus99 = False
        print(f"    {col}에 -99 잔존: {(df[col]==-99).sum():,}")
check("수치형 컬럼 -99 전부 제거", no_minus99)

# 문자형 '-99' 완전 제거
no_str_minus99 = True
for col in df.select_dtypes(include=[object]).columns:
    try:
        if (df[col] == "-99").sum() > 0 or (df[col] == "-99.0").sum() > 0:
            no_str_minus99 = False
            print(f"    {col}에 '-99' 잔존")
    except:
        pass
check("문자형 컬럼 '-99' 전부 제거", no_str_minus99)

# WGRADE: A/B/C/D만
check("WGRADE A/B/C/D만 존재",
      set(df['WGRADE'].dropna().unique()) <= {"A","B","C","D"})

# COST_AMT: 0 제거
check("COST_AMT 0 없음 (min > 0)", (df['COST_AMT'].dropna() > 0).all())

# area: 0 제거 (병합 버그 수정 확인)
for col in ["C2023","C2024","C2025","AREA"]:
    check(f"{col} 0 없음 (버그 수정 확인)", (df[col] == 0).sum() == 0)

# =====================================================
# 섹션 1. 기본 구조 (행 수 변동 / 무결성)
# =====================================================
print("\n[섹션 1] 기본 구조")
check("행 수 = 원본(2,408,699)", len(df) == len(raw_train))
check("열 수 = 44", df.shape[1] == 44)
check("CATTLE_NO 중복 없음", df["CATTLE_NO"].duplicated().sum() == 0)
check("CATTLE_NO 결측 없음", df["CATTLE_NO"].isnull().sum() == 0)
check("CATTLE_NO 집합 원본과 동일", set(df["CATTLE_NO"]) == set(raw_train["CATTLE_NO"]))

# =====================================================
# 섹션 2. 전처리 무손상 (전처리 안 한 컬럼 원본 대조)
# =====================================================
print("\n[섹션 2] 전처리 무손상 (원본 대조)")

# 전처리 안 한 수치형: WEIGHT, AGE는 원본과 완전 일치해야
m = df[["CATTLE_NO","WEIGHT","AGE"]].merge(
    raw_train[["CATTLE_NO","WEIGHT","AGE"]], on="CATTLE_NO", suffixes=("_now","_raw"))
check("WEIGHT 원본과 완전 일치 (변조 없음)", (m["WEIGHT_now"] == m["WEIGHT_raw"]).all())
check("AGE 원본과 완전 일치 (변조 없음)", (m["AGE_now"] == m["AGE_raw"]).all())

# 전처리 안 한 범주형 원본 대조
mc = df[["CATTLE_NO","sido","sigungu","JUDGE_SEX","LAST_GRADE"]].merge(
    raw_train[["CATTLE_NO","sido","sigungu","JUDGE_SEX","LAST_GRADE"]],
    on="CATTLE_NO", suffixes=("_now","_raw"))
check("sido 원본과 일치", (mc["sido_now"] == mc["sido_raw"]).all())
check("sigungu 원본과 일치", (mc["sigungu_now"] == mc["sigungu_raw"]).all())
check("JUDGE_SEX 원본과 일치", (mc["JUDGE_SEX_now"] == mc["JUDGE_SEX_raw"]).all())
check("LAST_GRADE 원본과 일치", (mc["LAST_GRADE_now"] == mc["LAST_GRADE_raw"]).all())

# =====================================================
# 섹션 3. 전처리 결측 수 검증 (의도한 변환만 발생)
# =====================================================
print("\n[섹션 3] 전처리 결측 수 검증 (원본 -99 대조)")

# 육질 컬럼: 원본 -99 개수 = 현재 결측 (역산 롤백했으므로 동일)
for col in ["BACKFAT","REA","WINDEX","INSFAT","YUKSAK","FATSAK","TISSUE","GROWTH"]:
    raw_99 = (raw_train[col] == -99).sum()
    now_null = df[col].isnull().sum()
    check(f"{col} 결측({now_null:,}) = 원본 -99({raw_99:,})", now_null == raw_99)

# WGRADE
raw_wg99 = raw_train["WGRADE"].astype(str).isin(["-99","-99.0"]).sum()
check(f"WGRADE 결측({df['WGRADE'].isnull().sum():,}) = 원본 -99({raw_wg99:,})",
      df["WGRADE"].isnull().sum() == raw_wg99)

# COST_AMT: 원본 -99 + 0원 1개
raw_cost99 = (raw_train["COST_AMT"] == -99).sum()
check(f"COST_AMT 결측({df['COST_AMT'].isnull().sum():,}) = 원본 -99({raw_cost99:,}) + 0원(1)",
      df["COST_AMT"].isnull().sum() == raw_cost99 + 1)

# death_count: death_df 없는 농장 소속 행
death_farms = set(raw_death["FARM_UNIQUE_NO"])
exp_death_null = (~df["FARM_UNIQUE_NO"].isin(death_farms)).sum()
check(f"death_count 결측({df['death_count'].isnull().sum():,}) = death_df 없는 농장({exp_death_null:,})",
      df["death_count"].isnull().sum() == exp_death_null)

# lineage 6개(KPN 제외): 원본 미등록 개체
lineage_cattle = set(raw_lineage["CATTLE_NO"])
exp_lin_null = (~df["CATTLE_NO"].isin(lineage_cattle)).sum()
for col in ["FATHER_CATTLE_NO","MOTHER_ANIMAL_NO","F_GMOTHER_ANIMAL_NO",
            "F_GFATHER_CATTLE_NO","M_GMOTHER_ANIMAL_NO","M_GFATHER_CATTLE_NO"]:
    check(f"{col} 결측({df[col].isnull().sum():,}) = 미등록({exp_lin_null:,})",
          df[col].isnull().sum() == exp_lin_null)

# KPN_NO: 미등록 + '-99' 변환 3,570
check(f"KPN_NO 결측({df['KPN_NO'].isnull().sum():,}) = 미등록({exp_lin_null:,}) + '-99'(3,570)",
      df["KPN_NO"].isnull().sum() == exp_lin_null + 3570)

# =====================================================
# 섹션 4. area 결측 행단위 검증 (버그 수정 정확성)
# =====================================================
print("\n[섹션 4] area 결측 행단위 검증")

area_farms = set(raw_area["FARM_UNIQUE_NO"])
for col in ["C2023","C2024","C2025","AREA"]:
    null_rows = df[df[col].isnull()]
    not_in = len(null_rows[~null_rows["FARM_UNIQUE_NO"].isin(area_farms)])
    in_null = len(null_rows[null_rows["FARM_UNIQUE_NO"].isin(area_farms)])
    total = df[col].isnull().sum()
    check(f"{col} 결측({total:,}) = 농장없음({not_in:,}) + 원본-99({in_null:,})",
          not_in + in_null == total)

# =====================================================
# 섹션 5. 데이터 타입
# =====================================================
print("\n[섹션 5] 데이터 타입")

check("BIRTH_YMD datetime", pd.api.types.is_datetime64_any_dtype(df["BIRTH_YMD"]))
check("ABATT_DATE datetime", pd.api.types.is_datetime64_any_dtype(df["ABATT_DATE"]))
check("JUDGE_DATE datetime", pd.api.types.is_datetime64_any_dtype(df["JUDGE_DATE"]))

num_cols = ["WEIGHT","BACKFAT","REA","AGE","COST_AMT","WINDEX","INSFAT",
            "YUKSAK","FATSAK","TISSUE","GROWTH","death_count",
            "C2023","C2024","C2025","AREA","days_total","days_양호",
            "days_주의","days_경고","days_위험","days_폐사",
            "rn_day_mean","ws_davg_mean","ta_min_mean"]
for c in num_cols:
    check(f"{c} 수치형", pd.api.types.is_numeric_dtype(df[c]))

cat_cols = ["sido","JUDGE_SEX","WGRADE","LAST_GRADE","CATTLE_NO","FARM_UNIQUE_NO"]
for c in cat_cols:
    check(f"{c} 문자형", df[c].dtype == object or pd.api.types.is_string_dtype(df[c]))

# =====================================================
# 섹션 6. BIRTH_YMD 무결성
# =====================================================
print("\n[섹션 6] BIRTH_YMD 무결성")

check("BIRTH_YMD 결측 없음", df["BIRTH_YMD"].isnull().sum() == 0)
check("BIRTH_YMD 1970-01-01 없음", (df["BIRTH_YMD"] == "1970-01-01").sum() == 0)
check("BIRTH_YMD 최솟값 >= 1996", df["BIRTH_YMD"].min().year >= 1996)
check("BIRTH_YMD <= ABATT_DATE", (df["BIRTH_YMD"] <= df["ABATT_DATE"]).all())
check("ABATT_DATE 범위 2023~2025", df["ABATT_DATE"].dt.year.between(2023, 2025).all())

# =====================================================
# 섹션 7. 결측치 현황 + 예상 검증
# =====================================================
print("\n[섹션 7] 결측치 현황")

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
summary = pd.DataFrame({"결측수": missing, "결측률(%)": missing_pct})
print(summary[summary["결측수"] > 0].sort_values("결측률(%)", ascending=False).to_string())

# 전체 NaN 컬럼 없는지
all_nan_cols = [c for c in df.columns if df[c].isnull().all()]
check(f"전체 NaN인 컬럼 없음 (발견: {all_nan_cols})", len(all_nan_cols) == 0)

# 육질 컬럼 결측이 전부 등외인지
q_null_grades = df[df["BACKFAT"].isnull()]["LAST_GRADE"].unique()
check("육질컬럼(BACKFAT) 결측 전부 등외", set(q_null_grades) <= {"등외"})

# =====================================================
# 섹션 8. 값 범위 / 논리
# =====================================================
print("\n[섹션 8] 값 범위 / 논리")

check("WEIGHT > 0", (df["WEIGHT"].dropna() > 0).all())
check("AGE > 0", (df["AGE"].dropna() > 0).all())
check("BACKFAT > 0", (df["BACKFAT"].dropna() > 0).all())
check("REA > 0", (df["REA"].dropna() > 0).all())
check("death_count >= 1 (결측 제외)", (df["death_count"].dropna() >= 1).all())
check("C2023 >= 1 (결측 제외)", (df["C2023"].dropna() >= 1).all())
check("AREA > 0 (결측 제외)", (df["AREA"].dropna() > 0).all())
check("days_total > 0", (df["days_total"] > 0).all())
check("등급별 일수 합 = 총일수",
      (df[["days_양호","days_주의","days_경고","days_위험","days_폐사"]].sum(axis=1)
       == df["days_total"]).all())
check("days_폐사 전부 0 (THI<98)", (df["days_폐사"] == 0).all())
check("ta_min_mean 범위(-30~30)", df["ta_min_mean"].between(-30, 30).all())
check("rn_day_mean >= 0", (df["rn_day_mean"] >= 0).all())
check("ws_davg_mean >= 0", (df["ws_davg_mean"] >= 0).all())

# =====================================================
# 섹션 9. 종속변수
# =====================================================
print("\n[섹션 9] 종속변수 LAST_GRADE")

print("\n등급 분포:")
print(df["LAST_GRADE"].value_counts().to_string())
check("LAST_GRADE 결측 없음", df["LAST_GRADE"].isnull().sum() == 0)
check("등외 포함 16개 등급", df["LAST_GRADE"].nunique() == 16)

# =====================================================
# 최종 요약
# =====================================================
print(f"검증 완료: 총 {len(results)}개 중 PASS {sum(results)}개, FAIL {len(results)-sum(results)}개")
print("=" * 70)

STEP 6 전처리 최종 검증

[섹션 0] 전처리 완료 검증
  [PASS] 수치형 컬럼 -99 전부 제거


/var/folders/6k/368vl_7x0678kz1qwgdkbwr40000gn/T/ipykernel_14615/1407109283.py:44: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=[object]).columns:


  [PASS] 문자형 컬럼 '-99' 전부 제거
  [PASS] WGRADE A/B/C/D만 존재
  [PASS] COST_AMT 0 없음 (min > 0)
  [PASS] C2023 0 없음 (버그 수정 확인)
  [PASS] C2024 0 없음 (버그 수정 확인)
  [PASS] C2025 0 없음 (버그 수정 확인)
  [PASS] AREA 0 없음 (버그 수정 확인)

[섹션 1] 기본 구조
  [PASS] 행 수 = 원본(2,408,699)
  [PASS] 열 수 = 44
  [PASS] CATTLE_NO 중복 없음
  [PASS] CATTLE_NO 결측 없음
  [PASS] CATTLE_NO 집합 원본과 동일

[섹션 2] 전처리 무손상 (원본 대조)
  [PASS] WEIGHT 원본과 완전 일치 (변조 없음)
  [PASS] AGE 원본과 완전 일치 (변조 없음)
  [PASS] sido 원본과 일치
  [PASS] sigungu 원본과 일치
  [PASS] JUDGE_SEX 원본과 일치
  [PASS] LAST_GRADE 원본과 일치

[섹션 3] 전처리 결측 수 검증 (원본 -99 대조)
  [PASS] BACKFAT 결측(4,567) = 원본 -99(4,567)
  [PASS] REA 결측(5,188) = 원본 -99(5,188)
  [PASS] WINDEX 결측(5,252) = 원본 -99(5,252)
  [PASS] INSFAT 결측(5,331) = 원본 -99(5,331)
  [PASS] YUKSAK 결측(5,269) = 원본 -99(5,269)
  [PASS] FATSAK 결측(4,743) = 원본 -99(4,743)
  [PASS] TISSUE 결측(5,320) = 원본 -99(5,320)
  [PASS] GROWTH 결측(9) = 원본 -99(9)
  [PASS] WGRADE 결측(4,814) = 원본 -99(4,814)
  [PASS] COST_AMT 결측(885,004) = 원본 -99(885,003) + 0원(1)
  [PA

In [3]:
df.to_csv("../../data/processed/2_preprocess/step6_preprocess.csv",
          index=False, encoding="utf-8-sig")
print(f"저장 완료: {df.shape}")

저장 완료: (2408699, 44)
